<a href="https://colab.research.google.com/github/google-research/population-dynamics/blob/master/notebooks/pdfm_global_embeddings_example.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##### Copyright 2024 Google LLC. Licensed under the Apache License, Version 2.0 (the "License");

In [ ]:
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License

# Data Preparation

## Load the embeddings -

 The following cells in this notebook will access the PD-Foundations embeddings directly from a BigQuery table.
⚠️ Important: To run these cells successfully, you must first obtain access to the embeddings dataset via a BigQuery Listing. Please use this form to apply for research access or to join the Early Access Program waitlist.

Once your access is approved, ensure you are authenticated in this Colab with a Google account that has permission to query the BigQuery table...

In [2]:
PROJECT_ID = '<project_id>' # @param {type:"string"}

In [3]:
!pip install pandas-gbq

In [25]:
import pandas as pd
import pandas_gbq

countries = ['BE', 'CH', 'DE', 'ES', 'FR', 'IT', 'NL', 'PT']

# @markdown Specify the BigQuery dataset for the embeddings.
bigquery_dataset = 'pdfm_embeddings_global' # @param {type: "string"}

embeddings = []
for country in countries:
  bigquery_table_id = f'{PROJECT_ID}.{bigquery_dataset}.{country.lower()}_embeddings_v1'

  query = f"SELECT * FROM `{bigquery_table_id}` WHERE region_type='postal_code'"
  df = pandas_gbq.read_gbq(query, project_id=PROJECT_ID, dialect='standard').set_index('place_name')

  embeddings.append(df)

embeddings_df = pd.concat(embeddings)


Downloading: 100%|██████████|
Downloading: 100%|██████████|
Downloading: 100%|██████████|
Downloading: 100%|██████████|
Downloading: 100%|██████████|
Downloading: 100%|██████████|
Downloading: 100%|██████████|
Downloading: 100%|██████████|


In [26]:
embeddings_df['country'] = embeddings_df['location_metadata'].apply(lambda x: x.get('country_code'))

## Fetch NUTS 3 region-level GDP from Eurostat via Data Commons

In [6]:
# Install datacommons_pandas
!pip install datacommons_pandas --upgrade --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.8/45.8 kB 2.1 MB/s eta 0:00:00


In [7]:
import datacommons_pandas as dc

europe = 'europe'
nuts3 = dc.get_places_in([europe], 'EurostatNUTS3')[europe]
print(len(nuts3))

1885


In [20]:
gdp = dc.build_multivariate_dataframe(nuts3, 'Amount_EconomicActivity_GrossDomesticProduction_Nominal_AsAFractionOf_Count_Person')
gdp.columns = ['gdp']
gdp['country'] = gdp.index.str[5:7]
print(gdp.shape)
gdp.head()

(1279, 2)


,gdp,country
place,,
nuts/AL011,3900,AL
nuts/AL012,5200,AL
nuts/AL013,3200,AL
nuts/AL014,3800,AL
nuts/AL015,3800,AL


## Aggregate embeddings to NUTS 3 level


In [21]:
#@title Get the NUTS to postal code crosswalk file from Eurostat.

import requests
import zipfile


def extract_zip(zip_path, extract_to_path):
    """Extracts the contents of a ZIP file to a specified directory."""
    try:
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(extract_to_path)
        print(f"ZIP file extracted successfully to: {extract_to_path}")
    except zipfile.BadZipFile:
        print(f"Error: {zip_path} is not a valid ZIP file.")
    except Exception as e:
        print(f"Error extracting ZIP file: {e}")

def download_zip(url, save_path):
    """Downloads a ZIP file from a given URL to a specified path."""
    try:
        response = requests.get(url, stream=True)
        response.raise_for_status()  # Raise an exception for bad status codes

        with open(save_path, 'wb') as f:
            for chunk in response.iter_content(chunk_size=8192):
                f.write(chunk)
        print(f"ZIP file downloaded successfully to: {save_path}")
    except requests.exceptions.RequestException as e:
        print(f"Error downloading ZIP file: {e}")

crosswalk_url = 'https://gisco-services.ec.europa.eu/tercet/NUTS-2024/pc2024_NUTS-2024_v3.0.zip'
crosswalk_path = '/content/crosswalk.zip'
extracted_path = '/content/crosswalk'
download_zip(crosswalk_url, crosswalk_path)
extract_zip(crosswalk_path, extracted_path)

ZIP file downloaded successfully to: /content/crosswalk.zip
ZIP file extracted successfully to: /content/crosswalk


In [22]:
crosswalk = []
def _read_crosswalk_file(country, version):
  return pd.read_csv(f'{extracted_path}/pc2024_{country}_NUTS-2024_v{version}.csv',
                     sep=';', quotechar='\'', dtype=str)
for country in countries:
  try:
    crosswalk.append(_read_crosswalk_file(country, '2.0'))
  except FileNotFoundError:
    crosswalk.append(_read_crosswalk_file(country, '1.0'))
crosswalk = pd.concat(crosswalk)
crosswalk.columns = ['nuts3', 'place_name']
crosswalk['country'] = crosswalk.nuts3.str[:2]
print(crosswalk.shape)
crosswalk.head()

(698560, 3)


,nuts3,place_name,country
0,BE334,4317,BE
1,BE213,2380,BE
2,BE212,2880,BE
3,BE345,6767,BE
4,BE336,4731,BE


In [27]:
#@title Aggregate embeddings
features = [f'feature{i}' for i in range(330)]

# Expand the 'features' list into separate columns in embeddings_df
expanded_features = pd.DataFrame(embeddings_df['features'].tolist(), index=embeddings_df.index)
expanded_features.columns = features # Assign the new feature column names

embeddings_df = embeddings_df.drop(columns=['features']).join(expanded_features)




In [28]:
embeddings_df.head()

,location_metadata,region_type,place_id,country,feature0,feature1,feature2,feature3,feature4,feature5,...,feature320,feature321,feature322,feature323,feature324,feature325,feature326,feature327,feature328,feature329
place_name,,,,,,,,,,,,,,,,,,,,,
TKCA 1ZZ,"{'latitude': None, 'administrative_area_level2...",postal_code,ChIJPdHNB07Uso4RAhm7S3Nd6dg,BE,0.036708,0.017967,6.868650e-03,0.011680,0.025451,0.000999,...,0.000000,0.000000,0.048830,0.000000e+00,-0.000006,0.030786,0.000000,0.027239,0.000000,-0.000540
TKCA 1ZZ,"{'latitude': None, 'administrative_area_level2...",postal_code,ChIJPdHNB07Uso4RAhm7S3Nd6dg,BE,0.021787,0.001020,-2.292800e-04,0.000394,0.027532,0.006281,...,-0.000027,-0.000001,0.051676,-4.115200e-04,-0.000019,0.023902,-0.000376,0.026810,-0.000076,-0.000752
TKCA 1ZZ,"{'latitude': None, 'administrative_area_level2...",postal_code,ChIJPdHNB07Uso4RAhm7S3Nd6dg,BE,0.001639,-0.000946,-3.181000e-05,-0.000660,0.000380,0.011372,...,-0.001482,-0.000805,0.053241,-1.065750e-03,-0.000409,0.027690,-0.001423,0.028556,-0.000893,0.003335
TKCA 1ZZ,"{'latitude': None, 'administrative_area_level2...",postal_code,ChIJPdHNB07Uso4RAhm7S3Nd6dg,BE,0.068700,0.042723,2.877136e-02,0.089345,0.037416,0.074868,...,-0.000001,0.000000,0.050039,-1.300000e-07,0.000000,0.027763,0.000000,0.023830,-0.001018,-0.000061
TKCA 1ZZ,"{'latitude': None, 'administrative_area_level2...",postal_code,ChIJPdHNB07Uso4RAhm7S3Nd6dg,BE,0.000218,0.012680,-4.000000e-08,0.026378,-0.001257,0.004679,...,-0.000359,-0.000021,0.052544,-6.449000e-05,-0.000522,0.023790,-0.000093,0.029028,-0.001343,0.006796


In [30]:
embeddings_df.reset_index(inplace=True)

In [31]:
left = embeddings_df.set_index(['country', 'place_name'])[features]
right = crosswalk.set_index(['country', 'place_name'])

nuts3_embeddings = left.join(right, how='inner').reset_index().groupby(['country', 'nuts3'])[features].mean().reset_index()
print(nuts3_embeddings.shape)
nuts3_embeddings.head()

(757, 332)


,country,nuts3,feature0,feature1,feature2,feature3,feature4,feature5,feature6,feature7,...,feature320,feature321,feature322,feature323,feature324,feature325,feature326,feature327,feature328,feature329
0,BE,BE100,0.027415,0.040365,0.007035,0.033325,0.023046,0.008483,0.046891,0.064028,...,0.038044,0.063429,0.026526,0.018097,0.050687,0.005212,0.018728,0.086760,0.034621,0.036841
1,BE,BE211,0.069471,0.061355,0.011206,0.053902,0.028123,0.010765,0.284750,0.135275,...,0.049238,0.035070,0.035848,0.013295,0.036046,0.005463,0.015460,0.056695,0.031208,0.059984
2,BE,BE212,0.080414,0.062900,0.012783,0.033978,0.028157,0.018503,0.290596,0.130453,...,0.052434,0.028554,0.040463,0.009524,0.031787,0.004636,0.013947,0.060821,0.024734,0.053120
3,BE,BE213,0.074563,0.066516,0.022871,0.034851,0.014666,0.020434,0.303516,0.136696,...,0.058878,0.011000,0.048658,0.018792,0.022154,0.005388,0.008838,0.063114,0.036450,0.050046
4,BE,BE223,0.052217,0.047608,0.022131,0.021478,0.011621,0.014052,0.204792,0.102713,...,0.066967,0.018845,0.051627,0.030410,0.012768,0.014479,0.007809,0.088366,0.043780,0.047660


# Training and Evaluation

## Train a multi-country model to predict for a new country

In [32]:
nuts3_embeddings.country.value_counts()

,count
country,
DE,400
IT,107
FR,96
ES,59
BE,44
NL,29
PT,22


In [33]:
nuts3_labels = gdp[gdp.country.isin(countries)].copy()
nuts3_labels['nuts3'] = nuts3_labels.index.str[5:]
nuts3_labels.country.value_counts()

,count
country,
DE,394
IT,107
FR,101
ES,59
BE,44
CH,27
NL,25
PT,11


In [34]:
data = nuts3_labels.set_index(['country', 'nuts3']).join(
    nuts3_embeddings.set_index(['country', 'nuts3']), how='inner').reset_index()
data.head()

,country,nuts3,gdp,feature0,feature1,feature2,feature3,feature4,feature5,feature6,...,feature320,feature321,feature322,feature323,feature324,feature325,feature326,feature327,feature328,feature329
0,BE,BE100,82100,0.027415,0.040365,0.007035,0.033325,0.023046,0.008483,0.046891,...,0.038044,0.063429,0.026526,0.018097,0.050687,0.005212,0.018728,0.086760,0.034621,0.036841
1,BE,BE211,60200,0.069471,0.061355,0.011206,0.053902,0.028123,0.010765,0.284750,...,0.049238,0.035070,0.035848,0.013295,0.036046,0.005463,0.015460,0.056695,0.031208,0.059984
2,BE,BE212,55800,0.080414,0.062900,0.012783,0.033978,0.028157,0.018503,0.290596,...,0.052434,0.028554,0.040463,0.009524,0.031787,0.004636,0.013947,0.060821,0.024734,0.053120
3,BE,BE213,60400,0.074563,0.066516,0.022871,0.034851,0.014666,0.020434,0.303516,...,0.058878,0.011000,0.048658,0.018792,0.022154,0.005388,0.008838,0.063114,0.036450,0.050046
4,BE,BE223,29700,0.052217,0.047608,0.022131,0.021478,0.011621,0.014052,0.204792,...,0.066967,0.018845,0.051627,0.030410,0.012768,0.014479,0.007809,0.088366,0.043780,0.047660


In [35]:
from sklearn import ensemble
from sklearn import linear_model
from sklearn import metrics as skmetrics

label = 'gdp'
train_countries = ['DE', 'IT', 'ES', 'BE', 'NL', 'CH', 'PT']
test_countries = ['FR']
train_data = data[data.country.isin(train_countries)]
test_data = data[data.country.isin(test_countries)]

model = linear_model.Ridge()
model.fit(train_data[features], train_data[label])
y_pred = model.predict(test_data[features])
y_true = test_data[label]

def evaluate(y_true, y_pred):
  r2 = skmetrics.r2_score(y_true, y_pred)
  mae = skmetrics.mean_absolute_error(y_true, y_pred)
  mape = skmetrics.mean_absolute_percentage_error(y_true, y_pred)
  return pd.Series({
      'r2': r2,
      'mae': mae,
      'mape': mape,
  }, name='metrics').round(3)

evaluate(y_true, y_pred)

,metrics
r2,0.324
mae,6349.939
mape,0.155
